# Blood: gene-level count table

Collapses the site table of the case-control samples to gene-level UMI counts (multimapping features removed). Used for the quality control in fig. S4A (UMIs and genes detected per sample).

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/blood_samples/data_processing")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
# System imports
import os
from joblib import dump, load

# Numerical and stats
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd

# Load the data

In [ ]:
adata = sc.read_h5ad(
    f"{DATA_ROOT}/3utr/Blood/all_till_06Jul/results/results_blood_3UTR_till_06jul_peaks_bw12_known_meta.h5ad"
)
adata

In [ ]:
adata.obs

In [ ]:
adata.var

# Create gene count table

In [ ]:
adata.var["gene_name"] = np.where(adata.var.gene_name.isna(), adata.var.name, adata.var.gene_name)
adata.var

In [ ]:
adata.var.gene_name.sort_values().unique()[380:500]

In [ ]:
adata.var.gene_name = adata.var.gene_name.str.lstrip(",")

In [ ]:
adata.var.gene_name.sort_values().unique()[0:120]

## Exclude multimapping reads

In [ ]:
multimapping = adata[:, adata.var.gene_name.str.contains(",")]
multimapping.var.gene_name

In [ ]:
# what's the number of reads that we will be removing?
display(multimapping.X.sum())
# and the total amount we have without removing
display(adata.X.sum())
# fraction of multimapping reads:
multimapping.X.sum() / adata.X.sum() * 100

In [ ]:
# So, remove multimapping reads
adata = adata[:, ~adata.var.gene_name.str.contains(",")]
adata.X.sum() == (72206406 - 423497)

## Turn 3'UTR counts into gene-level counts

In [ ]:
x_sum = adata.to_df().T
x_sum["gene_name"] = adata.var.gene_name
x_sum = x_sum.groupby("gene_name").sum()
x_sum

In [ ]:
var_new = adata.var.groupby("gene_name").first()
var_new = var_new.drop(
    ["utr_size", "score", "strand", "start", "end", "utr_name", "transcript_id"], axis=1
)
var_new

In [ ]:
adata_counts = ad.AnnData(X=x_sum.T, obs=adata.obs, var=var_new)
adata_counts

In [ ]:
adata_counts.var

In [ ]:
adata_counts.obs

In [ ]:
adata_counts.to_df()

In [ ]:
adata_counts.write_h5ad(
    f"{DATA_ROOT}/3utr/Blood/all_till_06Jul/results/results_blood_3UTR_till_06jul_gene_counts.h5ad"
)